In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date
import re

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.indianbank.in/departments/deposit-rates/" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find("table")
    rows=table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[3:]:  #Skip heading
        data = i.find_all("td")
        row=['Indian Bank']
        row.extend([tr.text for tr in data])
        for i in range(0,3):
            row[i] = ' '.join(re.sub(r'[\n@$#*\xa0]', '', row[i].strip()).split())
        #print("row=",row)
        if row_count > 0: row.extend(['',str(float(row[2])+0.50) + '%',''])
        else: row.extend(['', row[2] + '%',''])
        row[2] += '%'
        for i in range(0,6):
            row[i] = ' '.join(re.sub(r'[\n@$#*\xa0]', '', row[i].strip()).split())
        if row_count == 14: 
            tax_saver_row = ['Indian Bank', 'Tax-saver Fixed Deposit', row[2], row[3], row[4], row[5]]
            list_of_interest_rates.append(tax_saver_row)
        elif row_count == 8: row[1] = '300 days'
        elif row_count == 10: row[1] = '400 days'
        list_of_interest_rates.append(row)
        row_count += 1
            
    #print(list_of_interest_rates)
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nIndian Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 20112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_20112024.csv
      bank_name                                             tenure gen_rate  \
0   Indian Bank                                  7 days to 14 days    2.80%   
1   Indian Bank                                 15 days to 29 days    2.80%   
2   Indian Bank                                 30 days to 45 days    3.00%   
3   Indian Bank                                 46 days to 90 days    3.25%   
4   Indian Bank                                91 days to 120 days    3.50%   
5   Indian Bank                               121 days to 180 days    3.85%   
6   Indian Bank                     181 days to less than 9 months    4.50%   
7   Indian Bank  9 months to less than 1 year other than 300 Days.    4.75%   
8   Indian Bank                                           300 days    7.05%   
9   Indian Bank                                             1 year    6.10%   
10  Indian Bank            